<a href="https://colab.research.google.com/github/sneyx123-github/CopilotStudioSamples/blob/master/DrStop_DnsMapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import auth
PROJECT_ID = "project-46810a95-b6e5-47e4-adb"

# Melde dich mit deinem Google-Konto an
auth.authenticate_user()

# Setze das Projekt für die gcloud CLI
!gcloud config set project {PROJECT_ID}


[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


In [ ]:
import time
from datetime import datetime

# WICHTIG: Nur die Subdomain verwenden, kein http:// oder ://
DOMAIN = "us.kommunikator-gmbh.com"
REGION = "europe-west1"

# 1. Altes Mapping löschen
print(f"🗑️ Lösche altes Mapping für {DOMAIN}...")
!gcloud beta run domain-mappings delete --domain {DOMAIN} --region {REGION} --quiet

time.sleep(10)

# 2. Mapping neu erstellen
print(f"🚀 Erstelle Mapping neu für {DOMAIN}...")
!gcloud beta run domain-mappings create --service gradio-service --domain {DOMAIN} --region {REGION}

# 3. Überwachungsschleife
print(f"⏳ Überwachung gestartet für {DOMAIN}. Prüfe alle 5 Minuten...")

while True:
    status_output = !gcloud beta run domain-mappings describe --domain {DOMAIN} --region {REGION} --format="yaml(status.conditions)"
    status_str = "\n".join(status_output)
    now = datetime.now().strftime('%H:%M:%S')

    if "type: Ready\n    status: 'True'" in status_str:
        print(f"✅ [{now}] ERFOLG! Deine App ist LIVE unter https://{DOMAIN}")
        break
    elif "message: Certificate issuance pending" in status_str:
        print(f"🔄 [{now}] Google prüft DNS... (Certificate issuance pending)")
    elif "reason: CertificatePending" in status_str:
        print(f"⏳ [{now}] Warte auf Zertifikat-Erstellung...")
    else:
        print(f"📡 [{now}] Status: {status_str[:100]}...") # Zeigt den Anfang des Status an

    time.sleep(300)


🗑️ Lösche altes Mapping für us.kommunikator-gmbh.com...
Mappings to [us.kommunikator-gmbh.com] now have been deleted.
🚀 Erstelle Mapping neu für us.kommunikator-gmbh.com...
Waiting for certificate provisioning. You must configure your DNS records for certificate issuance to begin.
NAME            RECORD TYPE  CONTENTS
gradio-service  A            216.239.32.21
gradio-service  A            216.239.34.21
gradio-service  A            216.239.36.21
gradio-service  A            216.239.38.21
gradio-service  AAAA         2001:4860:4802:32::15
gradio-service  AAAA         2001:4860:4802:34::15
gradio-service  AAAA         2001:4860:4802:36::15
gradio-service  AAAA         2001:4860:4802:38::15
⏳ Überwachung gestartet für us.kommunikator-gmbh.com. Prüfe alle 5 Minuten...
⏳ [14:56:01] Warte auf Zertifikat-Erstellung...
